# Main manuscript analysis

This notebook is the public-facing analysis workflow.

It uses only the four manuscript event-window files in
`data/manuscript_storm_events/`. The complete long-term monitoring archive is
not required and is not included in this repository.


## 1. Setup

Install the repository in editable mode once with `pip install -e .`, then run
the notebook from the repository root.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

from wq_preprocessing.config import DEFAULT_CONFIG
from wq_preprocessing.data_io import load_manuscript_events
from wq_preprocessing.experiment import run_fixed_rule_experiment
from wq_preprocessing.hydrologic_phase import classify_all_storms
from wq_preprocessing.temporal_aggregation import resample_all_storms_dict

# Find repository root whether notebook is launched from root or /analysis
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "manuscript_storm_events"
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\Users\abadrud\Downloads\UPLOAD_THIS_TO_GITHUB_high_frequency_wq_preprocessing\UPLOAD_THIS_TO_GITHUB_high_frequency_wq_preprocessing
Data directory: c:\Users\abadrud\Downloads\UPLOAD_THIS_TO_GITHUB_high_frequency_wq_preprocessing\UPLOAD_THIS_TO_GITHUB_high_frequency_wq_preprocessing\data\manuscript_storm_events


## 2. Load the four manuscript storm events


In [6]:
all_storms = load_manuscript_events(DATA_DIR)

for storm_id, item in all_storms.items():
    print(
        storm_id,
        item["metadata"]["site"],
        item["data"].index.min(),
        item["data"].index.max(),
        len(item["data"])
    )

1 MS2226 2022-03-08 06:10:00 2022-03-13 12:00:00 643
2 MS2226 2024-03-01 00:08:00 2024-03-12 23:51:00 1566
3 MS3359 2023-02-09 00:06:00 2023-02-14 23:58:00 788
4 MS2226 2024-02-10 00:06:00 2024-02-17 23:56:00 1051


## 3. Hydrologic-phase classification

The radar-derived distance signal is inverted and normalized internally so that
larger values represent higher water level. The resulting `Stage_smoothed`
series is used for the hydrologic-phase analysis and hydrology-informed
gap-filling methods.


In [7]:
classified_storms = classify_all_storms(all_storms)

for storm_id, item in classified_storms.items():
    print(storm_id, item["info"])


1 {'dt_minutes': 12.0, 'baseline': np.float64(0.06617778572706094), 'onset_threshold': np.float64(0.09087819710248579), 'end_threshold': np.float64(0.08161554283670147), 'n_episodes': 2, 'n_peaks': 3, 'peak_indices': [np.int64(75), np.int64(137), np.int64(482)]}
2 {'dt_minutes': 11.0, 'baseline': np.float64(0.038940809968848224), 'onset_threshold': np.float64(0.06627725856697907), 'end_threshold': np.float64(0.056026090342679996), 'n_episodes': 3, 'n_peaks': 3, 'peak_indices': [np.int64(90), np.int64(700), np.int64(1074)]}
3 {'dt_minutes': 11.0, 'baseline': np.float64(0.013651877133105807), 'onset_threshold': np.float64(0.045870307167235516), 'end_threshold': np.float64(0.03378839590443687), 'n_episodes': 2, 'n_peaks': 4, 'peak_indices': [np.int64(124), np.int64(152), np.int64(399), np.int64(413)]}
4 {'dt_minutes': 11.0, 'baseline': np.float64(0.02884615384615284), 'onset_threshold': np.float64(0.05192307692307596), 'end_threshold': np.float64(0.0432692307692298), 'n_episodes': 3, 'n_p

## 4. Primary fixed-rule experiment

This reproduces the primary five gap-filling methods and nine
denoising/correction methods using the fixed experiment configuration.

The additional Moving Mean, ΔH-weighted, and HPWI gap-filling methods were added
later in the study and are implemented in `src/wq_preprocessing/gap_filling.py`.
The archived result tables used in the paper should be placed in `results/`.


In [ ]:
raw_results = run_fixed_rule_experiment(
    all_storms=all_storms,
    config=DEFAULT_CONFIG,
    checkpoint_dir=RESULTS / "checkpoints",
    checkpoint_every=500,
    verbose=True,
)

raw_results.to_csv(RESULTS / "results_fixed_rule_raw.csv", index=False)
raw_results.head()


## 5. Temporal aggregation

The manuscript sensitivity analysis evaluates 30-min and 1-h temporal
aggregation relative to the native event series. The helper below reproduces
the event aggregation. Frequency-aware block-missingness helpers are provided
in `temporal_aggregation.py`.


In [ ]:
storms_30min = resample_all_storms_dict(all_storms, "30min")
storms_1h = resample_all_storms_dict(all_storms, "1h")

print("Native:", {k: len(v["data"]) for k,v in all_storms.items()})
print("30 min:", {k: len(v["data"]) for k,v in storms_30min.items()})
print("1 h:", {k: len(v["data"]) for k,v in storms_1h.items()})


## 6. Figure reproduction

All plotting code has deliberately been moved out of this notebook.

Use:

`figures/figure_reproduction.ipynb`

after the analysis result CSV files have been generated.
